# 1. 필요 라이브러리 불러오기

In [8]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain import hub
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
import time

# 2. Knowledge Base 구성을 위한 데이터 생성

- Chroma를 활용한 [3.2 LangChain과 Chroma를 활용한 RAG 구성](https://github.com/jasonkang14/inflearn-rag-notebook/blob/main/3.2%20LangChain%EA%B3%BC%20Chroma%EB%A5%BC%20%ED%99%9C%EC%9A%A9%ED%95%9C%20RAG%20%EA%B5%AC%EC%84%B1.ipynb)과 동일함
- Vector Database만 [Pinecone](https://www.pinecone.io/)으로 변경

In [3]:
# from langchain_community.document_loaders import Docx2txtLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax_docs/tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
# from dotenv import load_dotenv
# from langchain_openai import OpenAIEmbeddings

load_dotenv()
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [5]:
# import os
# from pinecone import Pinecone
# from langchain_pinecone import PineconeVectorStore

index_name = 'tax-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)


database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit.  [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
]

In [9]:
database = PineconeVectorStore(
    index_name=index_name,
    embedding=embedding
)

# 2. 재시도 함수 (Chroma와 동일하게 사용)
def add_documents_with_retry(database, documents, max_retries=5):
    for attempt in range(max_retries):
        try:
            database.add_documents(documents)
            return True
        except Exception as e:
            error_message = str(e)
            # Pinecone 및 OpenAI API 사용량 초과 에러(429, RESOURCE_EXHAUSTED, Rate limit) 감지
            if (
                "429" not in error_message
                and "RESOURCE_EXHAUSTED" not in error_message
                and "rate_limit" not in error_message.lower()
            ):
                raise
            else:
                wait_seconds = (2 ** attempt) * 10
                print(f"사용량 제한(Rate Limit) 발생, {wait_seconds}초 후 재시도...")
                time.sleep(wait_seconds)

    return False

# 3. 배치 단위로 나누어 저장 실행
batch_size = 5

for start in range(0, len(document_list), batch_size):
    batch = document_list[start : start + batch_size]

    success = add_documents_with_retry(
        database=database,
        documents=batch
    )

    if not success:
        print(f"❌ {start}번째 청크부터 저장 실패")
        break

    print(f"✅ {start + 1}번 ~ {start + len(batch)}번 청크 저장 완료")
    
    # 파인콘 및 임베딩 API 쿼타 보호를 위해 5초 대기
    time.sleep(5)

✅ 1번 ~ 5번 청크 저장 완료
✅ 6번 ~ 10번 청크 저장 완료
✅ 11번 ~ 15번 청크 저장 완료
✅ 16번 ~ 20번 청크 저장 완료
사용량 제한(Rate Limit) 발생, 10초 후 재시도...
사용량 제한(Rate Limit) 발생, 20초 후 재시도...
✅ 21번 ~ 25번 청크 저장 완료
✅ 26번 ~ 30번 청크 저장 완료
✅ 31번 ~ 35번 청크 저장 완료
✅ 36번 ~ 40번 청크 저장 완료
✅ 41번 ~ 45번 청크 저장 완료
사용량 제한(Rate Limit) 발생, 10초 후 재시도...
사용량 제한(Rate Limit) 발생, 20초 후 재시도...
✅ 46번 ~ 50번 청크 저장 완료
✅ 51번 ~ 55번 청크 저장 완료
✅ 56번 ~ 60번 청크 저장 완료
✅ 61번 ~ 65번 청크 저장 완료
✅ 66번 ~ 70번 청크 저장 완료
사용량 제한(Rate Limit) 발생, 10초 후 재시도...
사용량 제한(Rate Limit) 발생, 20초 후 재시도...
✅ 71번 ~ 75번 청크 저장 완료
✅ 76번 ~ 80번 청크 저장 완료
✅ 81번 ~ 85번 청크 저장 완료
✅ 86번 ~ 90번 청크 저장 완료
✅ 91번 ~ 95번 청크 저장 완료
사용량 제한(Rate Limit) 발생, 10초 후 재시도...
사용량 제한(Rate Limit) 발생, 20초 후 재시도...
✅ 96번 ~ 100번 청크 저장 완료
✅ 101번 ~ 105번 청크 저장 완료
✅ 106번 ~ 110번 청크 저장 완료
✅ 111번 ~ 115번 청크 저장 완료
✅ 116번 ~ 120번 청크 저장 완료
사용량 제한(Rate Limit) 발생, 10초 후 재시도...
✅ 121번 ~ 125번 청크 저장 완료
✅ 126번 ~ 130번 청크 저장 완료
✅ 131번 ~ 135번 청크 저장 완료
✅ 136번 ~ 140번 청크 저장 완료
✅ 141번 ~ 145번 청크 저장 완료
사용량 제한(Rate Limit) 발생, 10초 후 재시도...
사용량 제한(Rate Lim

In [16]:
query = '연봉 5천만원인 거주자의 종합소득세는?'

# 3. 답변 생성을 위한 Retrieval

- `RetrievalQA`에 전달하기 위해 `retriever` 생성
- `search_kwargs` 의 `k` 값을 변경해서 가져올 문서의 갯수를 지정할 수 있음
- `.invoke()` 를 호출해서 어떤 문서를 가져오는지 확인 가능

In [10]:
retriever = database.as_retriever(search_kwargs={'k': 4})
# retriever.invoke(query)

# 4. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [15]:
# from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

/Users/jeong-yujin/.pyenv/versions/inflearn-llm-application-ver2/lib/python3.10/site-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [12]:
# from langchain_openai import ChatOpenAI
llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash')

# 5. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [13]:
# from langchain.chains import RetrievalQA


qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

In [17]:
# LangChain 권장사항에 따라 강의 코드와 다르게 `.invoke()`를 사용합니다
ai_message = qa_chain.invoke({"query": query})

In [20]:
ai_message

{'query': '연봉 5천만원인 거주자의 종합소득세는?',
 'result': '제공된 문서에는 연봉 5천만원에 대한 구체적인 근로소득공제액 및 세율표 등의 정보가 누락되어 있어 정확한 종합소득세를 알 수 없습니다. 따라서 제공된 정보만으로는 해당 질문에 답변할 수 없습니다.'}